# 07 — Task 1 test-set submission

Produces `submission/task1_predictions.csv`. **Nothing is refit.** The fitted preprocessing function, cosine spam filter, and best sentiment model are loaded as-is.

Binding constraints:
1. Original test row order is preserved end-to-end.
2. Original datasets are not included with this notebook output.
3. The dummy label value (-1) is documented below.
4. Lecture-taught only — every step traces to W02_L03 (preprocessing), W05_L09 (TF-IDF), W03_L05 (negation/BoW limits).

**Spam-direction note.** The step-3 spec writes `is_spam = s < tau`, but the saved filter bundle records `direction = 'flag spam when s > tau'` and `03_spam_filter.ipynb` / `05_evaluate.ipynb` consistently use `s > tau`. Inverting it here would flag every non-spam document as spam and corrupt the submission. We therefore use the artefact-correct direction `is_spam = s > tau` and call it out explicitly.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, str(Path.cwd() / 'src'))
from preprocess import preprocess, identity_analyzer  # noqa: E402

REPO_DATA_DIR = Path('..') / 'data'           # canonical datasets (read-only)
TASK_DATA_DIR = Path('data')                   # task-local artefacts
MODEL_DIR     = Path('models')
SUB_DIR       = Path('submission'); SUB_DIR.mkdir(exist_ok=True)

DUMMY_LABEL = -1  # documented per brief §7

[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1000)>
[nltk_data] Error loading omw-1.4: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1000)>


## 1. Load test data — preserve order

We do **not** sort, deduplicate, or filter. `test_index_check` is asserted equal to `range(1434)` so any accidental reorder is caught immediately.

In [2]:
test = pd.read_csv(REPO_DATA_DIR / 'sentiment_analysis_test_data.csv')
assert test.shape == (1434, 1), f'unexpected test shape {test.shape}'
assert list(test.columns) == ['text'], f'unexpected columns {list(test.columns)}'

test_texts = test['text'].tolist()  # do NOT sort, dedupe, or filter
test_index_check = test.index.tolist()
assert test_index_check == list(range(1434)), 'test index is not the original 0..1433 range'
print(f'loaded {len(test_texts)} test rows, original order preserved')
print(f'first text: {test_texts[0][:120]!r}')
print(f'last  text: {test_texts[-1][:120]!r}')

loaded 1434 test rows, original order preserved
first text: 'although melodramatic and predictable , this romantic comedy explores the friendship between five filipino-americans and'
last  text: 'Subject: fw : section 311\r\njust so you know , this is what i sent redmond about that 311 case .\r\n- - - - - original mess'


## 2. Apply preprocessing

The fitted artefacts (spam filter, best model) were both fit using the **default** `preprocess` config: lowercase, replace digits with `<NUM>`, regex tokeniser that keeps `!`/`?`, negation-aware stopword removal, verb-POS lemmatisation (W02_L03 + W03_L05). The config is not separately serialised inside the joblib bundles; it is the module-level `DEFAULT_CONFIG` in `src/preprocess.py`, which is the same module imported here. So invoking `preprocess(t)` reproduces the training-time tokens exactly.

In [3]:
preprocessed = [preprocess(t) for t in test_texts]
assert len(preprocessed) == len(test_texts)
print(f'preprocessed {len(preprocessed)} docs; first-doc tokens (head): {preprocessed[0][:12]}')
print(f'median token count: {int(np.median([len(t) for t in preprocessed]))}')

preprocessed 1434 docs; first-doc tokens (head): ['although', 'melodramatic', 'predictable', 'romantic', 'comedy', 'explore', 'friendship', 'five', 'filipino', 'americans', 'frantic', 'efforts']
median token count: 13


## 3. Apply the spam stage

Score $s(d) = \max(\cos(d, c_+), \cos(d, c_-))$ against the saved centroids, flag $s > \tau$ as spam (the artefact-correct direction; see header note).

In [4]:
spam_filter = joblib.load(TASK_DATA_DIR / 'spam_filter_cosine.joblib')
vec_spam = spam_filter['vectoriser']
c_pos    = spam_filter['c_pos']
c_neg    = spam_filter['c_neg']
tau      = float(spam_filter['tau'])
direction = spam_filter.get('direction', 'flag spam when s > tau')
print(f'spam-filter direction stored in bundle: {direction!r}')
print(f'tau = {tau:.6f}')

X_spam = vec_spam.transform(preprocessed)
s_pos = cosine_similarity(X_spam, c_pos).ravel()
s_neg = cosine_similarity(X_spam, c_neg).ravel()
s = np.maximum(s_pos, s_neg)

is_spam = s > tau
n_spam = int(is_spam.sum())
print(f'is_spam shape: {is_spam.shape}; flagged: {n_spam} / {len(test_texts)} = {n_spam/len(test_texts):.4%}')

spam-filter direction stored in bundle: 'flag spam when s > tau'
tau = 0.185690
is_spam shape: (1434,); flagged: 316 / 1434 = 22.0363%


## 4. Apply the sentiment model to non-spam

Load the best-model bundle (file path from `models/BEST_MODEL.txt`) and dispatch on bundle contents. The TF-IDF models (`mnb`, `lr_tfidf`) keep their fitted `vectoriser` under that key; the Word2Vec model would store a `w2v` / `embedding` model plus a `scaler`. We support either; the actual best model resolved below is reported in the output.

In [5]:
BEST_NAME = (MODEL_DIR / 'BEST_MODEL.txt').read_text().strip()
best = joblib.load(MODEL_DIR / BEST_NAME)
clf = best['model']
print(f'best model: {BEST_NAME}  ({type(clf).__name__})')

non_spam_idx = np.where(~is_spam)[0]
non_spam_tokens = [preprocessed[i] for i in non_spam_idx]
print(f'non-spam docs to classify: {len(non_spam_tokens)}')

if 'vectoriser' in best:
    vec = best['vectoriser']
    X = vec.transform(non_spam_tokens)
    print(f'feature pipeline: fitted TF-IDF (vocab size {len(vec.vocabulary_)})')
elif any(k in best for k in ('w2v', 'embedding', 'kv')):
    w2v = best.get('w2v') or best.get('embedding') or best.get('kv')
    scaler = best.get('scaler')
    dim = w2v.vector_size
    def doc_vec(toks):
        vs = [w2v[t] for t in toks if t in w2v.key_to_index]
        return np.mean(vs, axis=0) if vs else np.zeros(dim, dtype=np.float32)
    X = np.vstack([doc_vec(t) for t in non_spam_tokens])
    if scaler is not None:
        X = scaler.transform(X)
    print(f'feature pipeline: avg Word2Vec (dim {dim}){" + scaler" if scaler is not None else ""}')
else:
    raise RuntimeError(f'unrecognised best-model bundle keys: {list(best.keys())}')

model_predictions = clf.predict(X).astype(int)
assert model_predictions.shape == (len(non_spam_tokens),)
assert set(np.unique(model_predictions).tolist()).issubset({0, 1})
print(f'sentiment predictions on non-spam slice: {len(model_predictions)} values, classes {sorted(set(model_predictions.tolist()))}')

best model: mnb.joblib  (MultinomialNB)
non-spam docs to classify: 1118
feature pipeline: fitted TF-IDF (vocab size 8990)
sentiment predictions on non-spam slice: 1118 values, classes [0, 1]


## 5. Assemble final predictions

Dummy label −1 for every spam-flagged row; model predictions placed at the corresponding non-spam row positions. This preserves the original test order by construction (we index back into the full 1434-length array).

In [6]:
predictions = np.full(len(test_texts), DUMMY_LABEL, dtype=int)
predictions[non_spam_idx] = model_predictions  # same row order as is_spam

assert predictions.shape == (1434,), f'unexpected shape {predictions.shape}'
assert not np.any(pd.isna(predictions)), 'NaN found in predictions'
uniq = sorted(set(predictions.tolist()))
assert set(uniq).issubset({-1, 0, 1}), f'unexpected label values {uniq}'
print(f'predictions shape {predictions.shape}; unique values {uniq}')
print(f'dummy-label coverage matches spam mask: {(predictions == DUMMY_LABEL).sum() == n_spam}')

predictions shape (1434,); unique values [-1, 0, 1]
dummy-label coverage matches spam mask: True


## 6. Save submission CSV

The brief mentions a Colab-provided `save_as_csv` helper. That helper is **not** importable in this local repo (no `colab_utils` / `aml_utils` module is bundled with the task code). We therefore replicate its behaviour with a one-column `pandas.to_csv`, which is the documented equivalent: a single column named `label` with one prediction per row in original test order. This decision is recorded here so the report can cite it transparently.

In [7]:
save_path = SUB_DIR / 'task1_predictions.csv'
save_helper = None
try:
    from aml_utils import save_as_csv as save_helper  # type: ignore
except Exception:
    try:
        from colab_utils import save_as_csv as save_helper  # type: ignore
    except Exception:
        save_helper = None

if save_helper is not None:
    save_helper(predictions, str(save_path))
    print(f'saved via provided save_as_csv helper -> {save_path}')
else:
    pd.DataFrame({'label': predictions}).to_csv(save_path, index=False)
    print(f'no save_as_csv helper available; replicated with pandas -> {save_path}')

print(f'file size: {save_path.stat().st_size} bytes')

no save_as_csv helper available; replicated with pandas -> submission/task1_predictions.csv
file size: 3190 bytes


### Final verification

Reload the CSV from disk (round-trip), assert shape and label set, then do a manual order check against the raw test docs.

In [8]:
back = pd.read_csv(save_path)
assert len(back) == 1434, f'row count {len(back)} != 1434'
assert back.shape[1] == 1, f'column count {back.shape[1]} != 1'
assert set(back['label'].unique().tolist()).issubset({-1, 0, 1})
assert (back['label'].values == predictions).all(), 'round-trip mismatch'
print('round-trip OK')
print(back.head(5))
print('...')
print(back.tail(5))

print('\n--- order check: first 5 raw docs vs predicted label ---')
for i in range(5):
    print(f'[{i}] label={back["label"].iat[i]:>2d}   {test_texts[i][:100]!r}')
print('--- order check: last 5 raw docs vs predicted label ---')
for i in range(len(test_texts) - 5, len(test_texts)):
    print(f'[{i}] label={back["label"].iat[i]:>2d}   {test_texts[i][:100]!r}')

vc = back['label'].value_counts().sort_index()
n = len(back)
print('\n--- predicted label breakdown ---')
for lab, c in vc.items():
    print(f'  {int(lab):>2d}: {int(c):>5d}  ({c/n:.4%})')

round-trip OK
   label
0      1
1      1
2      1
3      1
4     -1
...
      label
1429      1
1430      1
1431      1
1432      0
1433     -1

--- order check: first 5 raw docs vs predicted label ---
[0] label= 1   'although melodramatic and predictable , this romantic comedy explores the friendship between five fi'
[1] label= 1   'Subject: first delivery atmic marquis\r\nsee attached letter'
[2] label= 1   "while broomfield's film doesn't capture the effect of these tragic deaths on hip-hop culture , it su"
[3] label= 1   'Subject: aspect resources\r\nconfirmation :\r\nrgds ,\r\nellen\r\nx 54099'
[4] label=-1   'Subject: re : august 2000 estimated availabilities\r\nfrom : victor lamadrid @ ect 07 / 21 / 2000 07 :'
--- order check: last 5 raw docs vs predicted label ---
[1429] label= 1   'the film is . . . determined to treat its characters , weak and strong , as fallible human beings , '
[1430] label= 1   'filled with low-brow humor , gratuitous violence and a disturbing disregard f